# ♟️ Chess Numba — Google Colab 雲端搜尋調參執行器 (History & Move Ordering Phase 4)

本 Notebook 用於執行 **Phase 4：歷史表反饋與走步排序啟發式體系（History Heuristics & Move Ordering）** 雲端 SPSA 調參與深度 SPRT 對抗驗收。

### 🎯 Phase 4 核心調參六軸
1. `LMR_HISTORY_SCALE`（當前基線 204）：LMR 歷史衰減係數，控制歷史評分對剪枝深度的微調力道。
2. `HISTORY_BONUS_SCALE`（當前基線 120）：安靜步剪枝歷史加分斜率（`bonus = min(SCALE * depth, CAP)`）。
3. `HISTORY_BONUS_CAP`（當前基線 1800）：安靜步歷史加分上限。
4. `HISTORY_MALUS_CAP`（當前基線 1600）：失敗走步歷史懲罰扣分上限。
5. `QUIET_ORDER_KILLER_1`（當前基線 400000）：第一殺手步（Killer Move 1）安靜步排序加分。
6. `QUIET_ORDER_COUNTER`（當前基線 300000）：應對步（Counter Move）安靜步排序加分。

### ⚡ 調參架構
* **L1 雲端 SPSA 探索**：80 輪 × 150 局 = 12,000 局 @ 50,000 nodes（`inprocess` 免重複 JIT 高速對抗）
* **Polyak-Ruppert 平滑取樣**：實證黃金比例 **`--tail 0.3`**（後 24 輪 / 3,600 局加權平均）
* **L2 深度獨立驗收**：600 局 @ 300,000 nodes，主開局庫 `data/openings.epd`，SPRT(0.0, 3.0)


In [4]:
# 1. 檢視 Colab 雲端硬體規格
!lscpu | grep "Model name\|Socket\|Thread\|NUMA\|CPU(s)"
!python3 --version


CPU(s):                                  8
On-line CPU(s) list:                     0-7
Model name:                              AMD EPYC 7B12
Thread(s) per core:                      2
Socket(s):                               1
NUMA node(s):                            1
NUMA node0 CPU(s):                       0-7
Python 3.13.16


In [5]:
# 2. 克隆/拉取最新代碼庫並安裝相依套件
import os
branch = "feat-initial-bitboard-setup"

if not os.path.exists("/content/chess_numba"):
    %cd /content
    !git clone -b {branch} https://github.com/b08203031/chess_numba.git

%cd /content/chess_numba
!git checkout -f {branch}
!git reset --hard
!git clean -fd
!rm -f tournament_analysis/l2_verify_history_p4_deep_tail03.json
!git pull origin {branch}
!pip install -q numba chess numpy


/content/chess_numba
Already on 'feat-initial-bitboard-setup'
Your branch is up to date with 'origin/feat-initial-bitboard-setup'.
HEAD is now at 31a4890 fix(colab): add git reset and clean to runner Cell 2 to prevent untracked file conflicts
From https://github.com/b08203031/chess_numba
 * branch            feat-initial-bitboard-setup -> FETCH_HEAD
Already up to date.


In [6]:
# 3. 測試環境與驗證 Phase 4 基準常數
import sys, json
sys.path.insert(0, "/content/chess_numba")
from tune_search.search_param_registry import get_default_params

phase4_params = [
    "LMR_HISTORY_SCALE",
    "HISTORY_BONUS_SCALE",
    "HISTORY_BONUS_CAP",
    "HISTORY_MALUS_CAP",
    "QUIET_ORDER_KILLER_1",
    "QUIET_ORDER_COUNTER"
]

print("🔍 Phase 4 歷史表與排序啟發式六軸基準值（Baseline）：")
print(json.dumps(get_default_params(phase4_params), indent=2))


🔍 Phase 4 歷史表與排序啟發式六軸基準值（Baseline）：
{
  "LMR_HISTORY_SCALE": 204,
  "HISTORY_BONUS_SCALE": 120,
  "HISTORY_BONUS_CAP": 1800,
  "HISTORY_MALUS_CAP": 1600,
  "QUIET_ORDER_KILLER_1": 400000,
  "QUIET_ORDER_COUNTER": 300000
}


In [7]:
# 4. 啟動 L1 SPSA Phase 4 歷史與排序六軸雲端對抗調參（目標 80 輪 x 150 局 = 12,000 局）
import os
import json

cpu_count = os.cpu_count() or 2
concurrency = max(1, cpu_count - 1 if cpu_count > 2 else cpu_count)
print(f"偵測到 Colab CPU 核心數: {cpu_count}，調參並行度設為: {concurrency}")

log_file = "tune_search/logs/history_p4_deep_l1.jsonl"
state_file = "tune_search/states/history_p4_deep_l1.json"
total_campaign_iters = 80

# 讀取存檔狀態以計算剩餘需要執行的迭代輪數（支援斷點續跑）
if os.path.exists(state_file):
    with open(state_file, "r", encoding="utf-8") as f:
        saved_state = json.load(f)
    done_iters = saved_state.get("iteration", 0)
    remaining_iters = max(0, total_campaign_iters - done_iters)
    resume_flag = "--resume"
    print(f"🔄 斷點續跑模式：已完成 {done_iters}/{total_campaign_iters} 輪，本次接續執行剩餘 {remaining_iters} 輪！")
else:
    done_iters = 0
    remaining_iters = total_campaign_iters
    resume_flag = ""
    print(f"🚀 全新開跑模式：計劃執行全量 {remaining_iters} 輪！")

if remaining_iters > 0:
    !python -m tune_search.search_tuner \
      --params LMR_HISTORY_SCALE,HISTORY_BONUS_SCALE,HISTORY_BONUS_CAP,HISTORY_MALUS_CAP,QUIET_ORDER_KILLER_1,QUIET_ORDER_COUNTER \
      --backend inprocess \
      --scale-mode integer \
      --iter {remaining_iters} \
      --games 150 \
      --nodes 50000 \
      --concurrency {concurrency} \
      --adjudicate \
      --log {log_file} \
      --state {state_file} \
      {resume_flag}
else:
    print("✅ Phase 4 的 80 輪調參已全部完成！請直接執行步驟 5 產出候選參數。\n")


偵測到 Colab CPU 核心數: 8，調參並行度設為: 7
🚀 全新開跑模式：計劃執行全量 80 輪！
[inprocess] openings=500 nodes=50000 concurrency=7 tt=16MB
[pool] allocate worker pair 1/7 (tt=16MB)...
[pool] allocate worker pair 2/7 (tt=16MB)...
[pool] allocate worker pair 3/7 (tt=16MB)...
[pool] allocate worker pair 4/7 (tt=16MB)...
[pool] allocate worker pair 5/7 (tt=16MB)...
[pool] allocate worker pair 6/7 (tt=16MB)...
[pool] allocate worker pair 7/7 (tt=16MB)...
[warmup] JIT nodes=5000 (may take several minutes first time)...
[warmup] done in 0.9s
[warmup] JIT nodes=1500 (may take several minutes first time)...
[warmup] done in 0.0s
[inprocess] touch worker 2/7
[warmup] JIT nodes=800 (may take several minutes first time)...
[warmup] done in 0.0s
[warmup] JIT nodes=800 (may take several minutes first time)...
[warmup] done in 0.0s
[inprocess] touch worker 3/7
[warmup] JIT nodes=800 (may take several minutes first time)...
[warmup] done in 0.0s
[warmup] JIT nodes=800 (may take several minutes first time)...
[warmup] done in 0

In [8]:
# 5. 分析 SPSA 調參成果，以 Polyak-Ruppert 後 30% 平均（tail 0.3）產出最佳候選參數
!python tune_search/summarize_run.py tune_search/logs/history_p4_deep_l1.jsonl --tail 0.3 --weight-mode uniform --out tune_search/candidate_history_p4_deep.json
!python tune_search/summarize_run.py tune_search/logs/history_p4_deep_l1.jsonl --tail 0.3 --weight-mode uniform --out tune_search/candidate_history_p4_tail03_uniform.json
!python tune_search/summarize_run.py tune_search/logs/history_p4_deep_l1.jsonl --tail 0.3 --weight-mode linear --out tune_search/candidate_history_p4_tail03_linear.json
print("\n=== Phase 4 等權最佳候選參數 JSON (tail 0.3 uniform) ===")
!cat tune_search/candidate_history_p4_deep.json
print("\n=== Phase 4 線性加權候選參數 JSON (tail 0.3 linear) ===")
!cat tune_search/candidate_history_p4_tail03_linear.json


iterations=80 games=12000 mean_score=0.5043
score sd observed=0.0319 noise-only expected=0.0305 ratio=1.05
  -> ratio ~1: batch scores are indistinguishable from sampling noise; this run carries (almost) no gradient signal. Use more games per batch / more batches before trusting any parameter movement.

param                             default      last  tail_avg moved%rng  drift_z
LMR_HISTORY_SCALE                     204     199.4     196.7     -1.85    -0.17  (noise)
HISTORY_BONUS_SCALE                   120     121.7     123.0      0.87     0.16  (noise)
HISTORY_BONUS_CAP                    1800    2058.3    2053.8     11.74     2.34
HISTORY_MALUS_CAP                    1600    1531.0    1537.3     -3.13    -0.62  (noise)
QUIET_ORDER_KILLER_1               400000  389272.2  390487.6     -2.15    -1.55  (noise)
QUIET_ORDER_COUNTER                300000  303916.9  302071.0      0.98     0.57  (noise)

averaged candidate (last 24 iterations, weight_mode=uniform):
--param LMR_HISTORY_

In [ ]:
# 6. 啟動 L2 深度 SPRT 實戰驗證（300,000 nodes, 600 局，主開局庫獨立驗證）
# 對照基準明確指定為 Phase 4 Baseline (tune_search/baseline_history_p4.json)
import os

cpu_count = os.cpu_count() or 2
concurrency = max(1, cpu_count - 1 if cpu_count > 2 else cpu_count)
print(f"啟動 Phase 4 L2 深度驗收，使用並行度: {concurrency}")

!python tools/run_search_param_match.py \
  --params-json tune_search/candidate_history_p4_deep.json \
  --baseline-json tune_search/baseline_history_p4.json \
  --nodes 300000 \
  --games 1000 \
  --openings data/openings.epd \
  --concurrency {concurrency} \
  --sprt 0,3 \
  --batch-games 50 \
  --adjudicate \
  --out tournament_analysis/l2_verify_history_p4_deep_tail03.json


啟動 Phase 4 L2 深度驗收，使用並行度: 7
search_param_match nodes=300000 games<=1000 concurrency=7 openings=2000 sprt=(0.0, 3.0)
baseline={'LMR_HISTORY_SCALE': 204, 'HISTORY_BONUS_SCALE': 120, 'HISTORY_BONUS_CAP': 1800, 'HISTORY_MALUS_CAP': 1600, 'QUIET_ORDER_KILLER_1': 400000, 'QUIET_ORDER_COUNTER': 300000}
candidate={'LMR_HISTORY_SCALE': 197, 'HISTORY_BONUS_SCALE': 123, 'HISTORY_BONUS_CAP': 2054, 'HISTORY_MALUS_CAP': 1537, 'QUIET_ORDER_KILLER_1': 390488, 'QUIET_ORDER_COUNTER': 302071}
[pool] allocate worker pair 1/7 (tt=16MB)...
[pool] allocate worker pair 2/7 (tt=16MB)...
[pool] allocate worker pair 3/7 (tt=16MB)...
[pool] allocate worker pair 4/7 (tt=16MB)...
[pool] allocate worker pair 5/7 (tt=16MB)...
[pool] allocate worker pair 6/7 (tt=16MB)...
[pool] allocate worker pair 7/7 (tt=16MB)...
[warmup] JIT nodes=5000 (may take several minutes first time)...
[warmup] done in 0.9s
[warmup] JIT nodes=1500 (may take several minutes first time)...
[warmup] done in 0.0s
[warmup] JIT nodes=800 (may take

In [ ]:
# 7. 策略 B：純 HISTORY_BONUS_CAP 專項隔離深度驗證（300,000 nodes, 600 局，純 Bonus Cap 2054 vs Baseline 1800）
# 隔離驗證：僅修改 HISTORY_BONUS_CAP 1800 -> 2054，其餘五軸（Killer 400k, Counter 300k, Malus 1600, Scale 120, LMR 204）嚴格鎖死 Baseline
import os, json
from pathlib import Path

# 自動確保候選檔存在（即使未 git pull 亦可直接執行）
bonuscap_candidate_path = Path("tune_search/candidate_history_p4_bonuscap2054.json")
bonuscap_candidate_path.parent.mkdir(parents=True, exist_ok=True)
bonuscap_params = {
  "LMR_HISTORY_SCALE": 204,
  "HISTORY_BONUS_SCALE": 120,
  "HISTORY_BONUS_CAP": 2054,
  "HISTORY_MALUS_CAP": 1600,
  "QUIET_ORDER_KILLER_1": 400000,
  "QUIET_ORDER_COUNTER": 300000
}
bonuscap_candidate_path.write_text(json.dumps(bonuscap_params, indent=2), encoding="utf-8")
print("🔍 策略 B 純 Bonus Cap 專項候選參數 (Cap=2054 vs Base 1800)：")
print(json.dumps(bonuscap_params, indent=2))

cpu_count = os.cpu_count() or 2
concurrency = max(1, cpu_count - 1 if cpu_count > 2 else cpu_count)
print(f"\n啟動純 HISTORY_BONUS_CAP 專項 L2 深度驗收，使用並行度: {concurrency}")

!python tools/run_search_param_match.py \
  --params-json tune_search/candidate_history_p4_bonuscap2054.json \
  --baseline-json tune_search/baseline_history_p4.json \
  --nodes 300000 \
  --games 600 \
  --openings data/openings.epd \
  --concurrency {concurrency} \
  --sprt 0,3 \
  --batch-games 50 \
  --adjudicate \
  --out tournament_analysis/l2_verify_history_p4_bonuscap2054.json
